# Rule-check model: GPU training on Kaggle (Phase 3)

A thin wrapper over the repository's versioned scripts (`ml/`), running exactly the Colab pipeline
with Kaggle paths; it adds no modelling code. On the **full official ContractNLI splits** it trains
`deberta_v3_base` (primary), `modernbert_base` (backbone ablation) and `deberta_v3_base_unseen4`
(4 hypotheses held out of training), evaluates them against the baselines with DEV-only choices,
exports ONNX fp32/int8 with parity checks, measures CPU latency and int8/fp32 TEST accuracy on this
machine, optionally runs the Anthropic zero-shot baseline, and packs a **verifiable results bundle**.

Read `ml/notebooks/KAGGLE.md` first. In short:
- Settings -> Accelerator **GPU T4 x2** (the pipeline uses one GPU, `cuda:0`), Internet **on**.
- Source without any token: the `git bundle` of the reviewed commit in a **private** Kaggle
  dataset attached as input (read-only under `/kaggle/input`). The notebook uses the one
  `*.bundle` it finds there and refuses none or several.
- Everything needed to resume or report goes to `/kaggle/working/compliance-runs` (saved with a
  version, up to Kaggle's 20 GB output limit). The source checkout, `.venv` and model caches live
  in scratch (`/kaggle/temp` or `/tmp`) and are not saved.
- Nothing is ever deleted from `/kaggle/working` or `/kaggle/input`; existing outputs are never
  overwritten. Every command's output is shown live and saved under `compliance-runs/logs/`.

No cell prints, logs or stores an API key.

## 0. Settings (the only cell to edit)

In [ ]:
# ruff: noqa: S101, S108, S603, S607, T201  (notebook: shell calls, prints, asserts, /tmp intended)
EXPECTED_COMMIT = ""  # full 40-character commit hash of the reviewed source (required)
# None: the ONE *.bundle file under /kaggle/input (the private source dataset); several are
# refused. Otherwise the exact path of a bundle under /kaggle/input.
SOURCE_BUNDLE = None
# None, or a previous version's output attached as input (a folder under /kaggle/input holding
# compliance-runs/): verified reusable parts are COPIED into this session's output (KAGGLE.md).
IMPORT_PREVIOUS_OUTPUT = None
# Training runs of this session. Evaluation, export and the results bundle run once all three
# runs are complete (in this session or imported).
TRAIN_RUNS = ("deberta_v3_base", "modernbert_base", "deberta_v3_base_unseen4")
# Real Anthropic API calls billed to your key (Kaggle secret ANTHROPIC_API_KEY); see KAGGLE.md.
RUN_LLM = False
OUTPUT_LIMIT_GB = 20  # Kaggle's saved-output limit for /kaggle/working

## 1. Kaggle paths and GPU

In [ ]:
import json
import os
import pathlib
import shutil
import subprocess
import sys

SCRATCH_MIN_GIB = 15  # .venv with CUDA torch, uv cache, Python download, pinned model files


def pick_scratch(candidates):
    """The first existing, writable directory: /kaggle/temp (not saved), else /tmp."""
    for path in map(pathlib.Path, candidates):
        if path.is_dir() and os.access(path, os.W_OK):
            return path
    raise RuntimeError(f"no writable scratch directory among {candidates}")


def inside(path, root):
    """Lexically inside root (or root itself), without following links."""
    path = pathlib.Path(os.path.abspath(path))
    root = pathlib.Path(os.path.abspath(root))
    return path == root or root in path.parents


def check_layout(input_root, working, runs, scratch, repo):
    """Inputs are only read; outputs only in RUNS; source, .venv and caches only in scratch."""
    problems = []
    if not inside(runs, working) or inside(working, runs):
        problems.append(f"RUNS {runs} must be a folder inside {working}")
    if not inside(repo, scratch) or inside(scratch, repo):
        problems.append(f"the source checkout {repo} must be a folder inside scratch {scratch}")
    if inside(scratch, working) or inside(working, scratch):
        problems.append(f"scratch {scratch} and {working} must not overlap (it would be saved)")
    for name, path in (("RUNS", runs), ("scratch", scratch)):
        if inside(path, input_root) or inside(input_root, path):
            problems.append(f"{name} {path} must not overlap the read-only inputs {input_root}")
    if problems:
        raise RuntimeError("unsafe layout:\n" + "\n".join(problems))


INPUT = pathlib.Path("/kaggle/input")  # attached datasets / previous outputs: read-only
WORKING = pathlib.Path("/kaggle/working")  # saved with a version (output limit)
RUNS = WORKING / "compliance-runs"
SCRATCH = pick_scratch(["/kaggle/temp", "/tmp"]) / "compliance-scratch"
REPO = SCRATCH / "compliance"
check_layout(INPUT, WORKING, RUNS, SCRATCH, REPO)
SCRATCH.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
# Downloads and caches stay in scratch, so a saved version holds no environment or model cache.
for var, sub in (
    ("UV_CACHE_DIR", "uv-cache"),
    ("UV_PYTHON_INSTALL_DIR", "uv-python"),
    ("PIP_CACHE_DIR", "pip-cache"),
    ("HF_HOME", "hf-home"),
    ("XDG_CACHE_HOME", "xdg-cache"),
    ("TORCH_HOME", "torch-home"),
):
    os.environ[var] = str(SCRATCH / sub)
# One GPU: the pipeline uses cuda:0 only (GPU T4 x2 offers two; the second stays idle).
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run(["nvidia-smi"], check=True)
print("kernel Python", sys.version.split()[0], "| runs:", RUNS, "| scratch:", SCRATCH)
for where, path in (("working (saved)", WORKING), ("scratch (not saved)", SCRATCH)):
    usage = shutil.disk_usage(path)
    print(f"{where}: {usage.free / 1024**3:.1f} GiB free of {usage.total / 1024**3:.1f} GiB")

## 2. Source code: the reviewed commit from the private bundle dataset

On the development machine: `git bundle create compliance.bundle HEAD main`, uploaded as a
**private** Kaggle dataset and attached to this notebook (KAGGLE.md). The cell verifies the bundle,
clones it into scratch (or, in the same session, fetches a new bundle into `FETCH_HEAD` only) and
checks out exactly `EXPECTED_COMMIT`, refusing any implementation change outside the generated
`ml/eval/results/`.

In [ ]:
import tempfile

# The pipeline itself rewrites ml/eval/results/ (data_report.json is tracked; evaluation adds
# new files), so a rerun in the same session excludes ONLY that generated subtree. Any other
# changed or untracked file still refuses to train.
GENERATED_RESULTS = "ml/eval/results"
BUNDLE_SEARCH_DEPTH = 4  # /kaggle/input/<dataset>/... levels searched for *.bundle


def find_bundles(input_root, max_depth=BUNDLE_SEARCH_DEPTH):
    """Every *.bundle file under input_root, at most max_depth folders deep, sorted."""
    input_root = pathlib.Path(input_root)
    found = []
    for root, dirs, files in os.walk(input_root, followlinks=True):
        if len(pathlib.Path(root).relative_to(input_root).parts) >= max_depth:
            dirs[:] = []
        dirs.sort()
        found += [pathlib.Path(root) / f for f in sorted(files) if f.endswith(".bundle")]
    return found


def source_bundle(input_root, explicit=None):
    """SOURCE_BUNDLE (a file under the read-only inputs), or the only *.bundle found there."""
    if explicit is not None:
        path = pathlib.Path(explicit)
        if not inside(path, input_root) or not path.is_file():
            raise RuntimeError(f"SOURCE_BUNDLE {explicit} is not a file under {input_root}")
        return path
    found = find_bundles(input_root)
    if len(found) != 1:
        listing = "\n".join(f"  {p}" for p in found) or "  (none)"
        raise RuntimeError(
            f"expected exactly one *.bundle under {input_root}, found {len(found)}:\n{listing}\n"
            "Attach only the private source dataset, or set SOURCE_BUNDLE to one exact path."
        )
    return found[0]


def git(repo, *args):
    """A git command in repo; on failure raises RuntimeError with git's own message."""
    out = subprocess.run(["git", "-C", str(repo), *args], capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed: {out.stderr.strip()}")
    return out.stdout


def implementation_changes(repo):
    """`git status` lines for changed/untracked files outside the generated results tree."""
    return subprocess.run(
        [
            "git",
            "-C",
            str(repo),
            "status",
            "--porcelain",
            "--untracked-files=all",
            "--",
            ".",
            f":(exclude){GENERATED_RESULTS}",
        ],
        capture_output=True,
        text=True,
        check=True,
    ).stdout


def has_commit(repo, commit):
    probe = ["git", "-C", str(repo), "cat-file", "-e", f"{commit}^{{commit}}"]
    return subprocess.run(probe, capture_output=True).returncode == 0


def verify_bundle(bundle, work_dir):
    """`git bundle verify` needs a repository: verify inside a fresh empty one."""
    check = pathlib.Path(tempfile.mkdtemp(prefix="bundle-check-", dir=work_dir))
    try:
        git(check, "init", "--quiet")
        git(check, "bundle", "verify", str(bundle))
    finally:
        shutil.rmtree(check)


def checkout_source(repo, commit, bundle):
    """Check out commit in repo from the verified bundle and return HEAD; never overwrites.

    A fresh session clones the bundle. In an existing checkout (same session) a missing commit
    is fetched from the bundle into FETCH_HEAD only (no branch update, reset or working-tree
    change), and the checkout happens only while the implementation is clean; git itself
    refuses to overwrite a changed generated file.
    """
    repo = pathlib.Path(repo)
    repo.parent.mkdir(parents=True, exist_ok=True)
    verify_bundle(bundle, repo.parent)
    if not repo.exists():
        subprocess.run(["git", "clone", "--quiet", str(bundle), str(repo)], check=True)
    elif not has_commit(repo, commit):
        refs = [
            line.split()[1] for line in git(repo, "bundle", "list-heads", str(bundle)).splitlines()
        ]
        git(repo, "fetch", "--quiet", str(bundle), *refs)  # FETCH_HEAD only
    if not has_commit(repo, commit):
        raise RuntimeError(f"{commit} is not in {bundle}")
    dirty = implementation_changes(repo)
    if dirty:
        raise RuntimeError(f"implementation changes in {repo}; refusing to check out:\n{dirty}")
    git(repo, "checkout", "--quiet", commit)
    return git(repo, "rev-parse", "HEAD").strip()


assert len(EXPECTED_COMMIT) == 40, "set EXPECTED_COMMIT to the full commit hash"
BUNDLE = source_bundle(INPUT, SOURCE_BUNDLE)
print("source bundle:", BUNDLE)
head = checkout_source(REPO, EXPECTED_COMMIT, BUNDLE)
dirty = implementation_changes(REPO)
assert head == EXPECTED_COMMIT, head
assert not dirty, dirty
print("training source:", head, "(implementation clean; generated results excluded)")
os.chdir(REPO)

## 3. Runtime helpers (from the verified checkout) and output preflight

`ml/notebooks/kaggle_runtime.py` (standard library only) runs every step as a child process with
live, logged output, budgets the saved output and copies with verification. The preflight
reports the saved output (`compliance-runs`) against the output limit separately from the space
actually free on the working and scratch filesystems, and the state of each training run.
With `RUN_LLM = True` it also checks now, before any GPU work, that the Kaggle secret
`ANTHROPIC_API_KEY` is configured (no API call; the value is not kept or shown).

In [ ]:
import importlib.util

_spec = importlib.util.spec_from_file_location(
    "kaggle_runtime", REPO / "ml" / "notebooks" / "kaggle_runtime.py"
)
rt = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rt)
LOGS = RUNS / "logs"
OUTPUT_LIMIT = int(OUTPUT_LIMIT_GB * rt.GB)
assert set(TRAIN_RUNS) <= set(rt.RUN_NAMES), TRAIN_RUNS


def stream(cmd, name, env=None):
    """cmd with its output shown live and logged under RUNS/logs (see kaggle_runtime.stream)."""
    return rt.stream(cmd, name, LOGS, env)


def run(*args, **env):
    """A repository module with the project interpreter PY (set by the environment cell)."""
    return rt.run_module(PY, LOGS, *args, **env)


def train(name, config):
    """One training run in RUNS/<name>; it resumes from its own complete checkpoint."""
    if name not in TRAIN_RUNS:
        print(f"{name}: not in TRAIN_RUNS for this session ({rt.run_status(RUNS / name)})")
        return
    assert_cuda()
    rt.require_training_room(RUNS, name, OUTPUT_LIMIT)
    run("ml.train.train", "--config", config, "--run-dir", str(RUNS / name))


if RUN_LLM:  # checked now, before hours of GPU work; the value is read again when needed
    from kaggle_secrets import UserSecretsClient

    rt.require_secret("ANTHROPIC_API_KEY", UserSecretsClient().get_secret)
rt.output_report(RUNS, OUTPUT_LIMIT, SCRATCH)
rt.require_free(SCRATCH, SCRATCH_MIN_GIB, "scratch (environment and model caches)")

## 4. Environment: the locked dependencies + CUDA PyTorch of the same version

`uv` is installed into scratch with the kernel's `python -m pip --target` (Kaggle's own Python
environment is left unchanged). `uv sync --frozen` creates the project's Python 3.11 `.venv` from
`uv.lock` (CPU torch 2.14.1); the torch wheel is then replaced by the **CUDA 12.6 build of the same
version** and the ML tool pins are added. All later commands call `.venv/bin/python` directly
(never `uv run` or `make`, which would re-sync the CPU torch), and `assert_cuda()` re-checks the
interpreter, the build and a real CUDA operation right before every GPU step.

In [ ]:
UV_TOOL = SCRATCH / "uv-tool"
if not (UV_TOOL / "bin" / "uv").is_file():
    stream(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            "--target",
            str(UV_TOOL),
            "uv==0.8.22",
        ],
        "install-uv",
    )
UV = str(UV_TOOL / "bin" / "uv")
stream([UV, "sync", "--frozen"], "uv-sync")
stream(
    [
        UV,
        "pip",
        "install",
        "--python",
        ".venv/bin/python",
        "--reinstall-package",
        "torch",
        "torch==2.14.1+cu126",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
    ],
    "cuda-torch",
)
stream(
    [UV, "pip", "install", "--python", ".venv/bin/python", "-r", "ml/requirements-tools.txt"],
    "ml-tools",
)
PY = str(REPO / ".venv/bin/python")
CUDA_PROBE = (
    "import json, sys, torch; x = torch.ones(2, device='cuda'); print(json.dumps({"
    "'python': list(sys.version_info[:2]), 'torch': torch.__version__, "
    "'cuda': torch.cuda.is_available(), 'devices': torch.cuda.device_count(), "
    "'name': torch.cuda.get_device_name(0), 'sum': float((x + x).sum())}))"
)


def assert_cuda():
    out = subprocess.run(
        [PY, "-c", CUDA_PROBE],
        capture_output=True,
        text=True,
        env={**os.environ, "PYTHONPATH": ""},
    )
    if out.returncode != 0:
        raise RuntimeError(f"CUDA check failed:\n{out.stderr[-3000:]}")
    info = json.loads(out.stdout.strip().splitlines()[-1])
    assert info["python"] == [3, 11], f"project Python {info['python']}, expected 3.11"
    assert info["torch"] == "2.14.1+cu126", f"wrong torch {info['torch']} (re-synced environment?)"
    assert info["cuda"], f"CUDA is not available: {info}"
    assert info["sum"] == 4.0, f"CUDA arithmetic failed: {info}"
    return info


print("CUDA ready:", assert_cuda())

## 5. Pinned artifacts (ContractNLI archive + BGE encoder; SHA-256 verified, into scratch)

In [ ]:
run("compliance_ingestion.embeddings", "fetch", ARTIFACT_CACHE_DIR=".cache")
run("compliance_seed.fetch", ARTIFACT_CACHE_DIR=".cache")

## 6. Optional: continue from a previous version's saved output

Leave `IMPORT_PREVIOUS_OUTPUT = None` for a fresh start. Otherwise attach the previous version's
output as an input and set the path of its `compliance-runs` folder under `/kaggle/input`. The
input is only read. Copied into this session's `compliance-runs` (never over an existing entry):
the prepared data, the baselines, the LLM response cache, complete runs with the current
training-code fingerprint and data version (without their 2.2 GB resume checkpoint), and
unfinished runs **only** with a complete resume checkpoint. Weights alone (for example the failed
Colab run's `checkpoints/best`) are never resumed or warm-started: that run starts fresh here.

In [ ]:
FINGERPRINT = subprocess.run(
    [PY, "-c", "from ml.common.provenance import training_code_fingerprint as f; print(f())"],
    capture_output=True,
    text=True,
    check=True,
    env={**os.environ, "PYTHONPATH": ""},
).stdout.strip()
print("training-code fingerprint:", FINGERPRINT)
if IMPORT_PREVIOUS_OUTPUT is not None:
    if not inside(IMPORT_PREVIOUS_OUTPUT, INPUT):
        raise RuntimeError("IMPORT_PREVIOUS_OUTPUT must be a read-only input under /kaggle/input")
    rt.import_previous_output(
        pathlib.Path(IMPORT_PREVIOUS_OUTPUT),
        RUNS,
        training_fingerprint=FINGERPRINT,
        limit_bytes=OUTPUT_LIMIT,
    )
for name in rt.RUN_NAMES:
    print(f"{name}: {rt.run_status(RUNS / name)}")

## 7. Processed data (official splits, segments, candidate ranking) and classic baselines

Prepared once and kept in `compliance-runs/data-full`, so every session and every run uses the
**same `data_version`** (a resumed run refuses a different one). The dense candidate ranking uses
the pinned BGE encoder on the GPU. The majority and TF-IDF baselines run on CPU on the same data.

In [ ]:
DATA_SAVED = RUNS / "data-full"  # canonical prepared data (same data_version every session)
DATA = REPO / ".cache/ml/data/full"
if DATA_SAVED.is_dir():
    rt.replace_scratch_copy(DATA_SAVED, DATA, SCRATCH)
    print("restored the prepared data from", DATA_SAVED)
else:
    assert_cuda()
    run("ml.data.prepare", "--device", "cuda", "--threads", "2")
stream(
    [
        PY,
        "-c",
        "from pathlib import Path; from ml.data.store import load_dataset; "
        "load_dataset(Path('.cache/ml/data/full'))",
    ],
    "verify-data",
    {"PYTHONPATH": ""},
)  # verifies every file hash
if not DATA_SAVED.exists():
    rt.copy_verified(DATA, DATA_SAVED)
manifest = json.loads((DATA / "manifest.json").read_text())
print("data_version", manifest["data_version"], manifest["counts"])
BASELINES = RUNS / "baselines"
if not (BASELINES / "classic_baselines.json").exists():
    run("ml.baselines.classic", "--out", str(BASELINES))

## 8. Training

Each run selects its checkpoint/epoch and label top-k on DEV only; TEST is predicted after that
selection is frozen. Checkpoints are written to `compliance-runs/<run>`: rerunning a cell in the
**same** session resumes from `checkpoints/last`; a new session resumes only if that folder is
still there (persistence) or was imported from a saved version (section 6). A different data
version, tokenizer, config or training code is refused on resume. Runs not listed in
`TRAIN_RUNS` are skipped.

In [ ]:
train("deberta_v3_base", "ml/train/configs/deberta_v3_base.yaml")

In [ ]:
train("modernbert_base", "ml/train/configs/modernbert_base.yaml")

In [ ]:
train("deberta_v3_base_unseen4", "ml/train/configs/deberta_v3_base_unseen4.yaml")

## 9. Evaluation, export, CPU latency and quantized TEST accuracy

Runs only when all three runs are complete. The serving backbone is chosen on DEV. Export writes
ONNX fp32 + int8 with parity checks to `compliance-runs`. The benchmark measures this machine's
CPU (recorded in the result file as `kaggle_cpu`). The int8/fp32 TEST accuracy run is required
for the phase and takes a while on CPU.

In [ ]:
STATUS = {name: rt.run_status(RUNS / name) for name in rt.RUN_NAMES}
READY = all(status == "complete" for status in STATUS.values())
if not READY:
    print("evaluation, export and the results bundle need three complete runs:", STATUS)
    print("save this version, then continue with IMPORT_PREVIOUS_OUTPUT (see KAGGLE.md)")


def evaluate(*extra):
    run(
        "ml.eval.evaluate",
        "--run-dir",
        str(RUNS / "deberta_v3_base"),
        "--ablation-run-dir",
        str(RUNS / "modernbert_base"),
        "--unseen-run-dir",
        str(RUNS / "deberta_v3_base_unseen4"),
        "--baselines-dir",
        str(BASELINES),
        *extra,
    )


if READY:
    evaluate()
    params = json.loads(pathlib.Path("ml/eval/results/inference_params.json").read_text())
    CHOSEN = pathlib.Path(params["run_dir"])
    ARTIFACT = RUNS / f"artifact-{CHOSEN.name}"
    if not ARTIFACT.exists():
        rt.require_output_room(RUNS, rt.FINAL_BYTES["export"], "ONNX export", OUTPUT_LIMIT)
    assert_cuda()
    run(
        "ml.export.export",
        "--run-dir",
        str(CHOSEN),
        "--params",
        "ml/eval/results/inference_params.json",
        "--model-eval",
        "ml/eval/results/model_eval.json",
        "--out",
        str(ARTIFACT),
    )
    THREADS = str(os.cpu_count() or 2)
    run(
        "ml.export.bench",
        "--artifact-dir",
        str(ARTIFACT),
        "--run-dir",
        str(CHOSEN),
        "--label",
        "kaggle_cpu",
        "--threads",
        THREADS,
        "--out",
        "ml/eval/results/latency_kaggle_cpu.json",
    )
    run(
        "ml.export.eval_onnx",
        "--artifact-dir",
        str(ARTIFACT),
        "--run-dir",
        str(CHOSEN),
        "--split",
        "test",
        "--threads",
        THREADS,
        "--out",
        "ml/eval/results/quantization_eval.json",
    )

## 10. Optional: the Anthropic zero-shot baseline (real API calls, billed to your key)

Leave `RUN_LLM = False` unless you have decided to pay for it. If enabled, the key is read from
the Kaggle secret `ANTHROPIC_API_KEY` (Add-ons -> Secrets, attached to this notebook) into the
child process's environment only (section 3 already checked that it is configured). The dry runs
print the prompt and a rough cost estimate. The response cache lives in `compliance-runs/llm-cache`
(the script's cache path links there), so every recorded attempt is in the saved output as soon as
it is written, and a rerun, also in a later session with the output imported, does not pay twice
for answered pairs.

In [ ]:
LLM_SAVED = RUNS / "llm-cache"  # canonical response cache, in the saved output
if RUN_LLM:
    # The script's cache path becomes a link to LLM_SAVED: each attempt it flushes is saved.
    rt.link_cache(REPO / ".cache/ml/llm", LLM_SAVED, SCRATCH)
    for split in ("dev", "test"):
        run("ml.baselines.llm", "--split", split, "--dry-run")
    from kaggle_secrets import UserSecretsClient

    key_env = rt.secret_env("ANTHROPIC_API_KEY", UserSecretsClient().get_secret)
    try:
        for split in ("dev", "test"):
            run("ml.baselines.llm", "--split", split, "--out", str(BASELINES), **key_env)
    finally:
        del key_env

## 11. Final evaluation and artifact parameters

Re-evaluates with the measured latency (and the LLM results if present: the cascade tau is then
selected on DEV), and refreshes the artifact manifest so the served parameters and version match
the final DEV selections (weights unchanged).

In [ ]:
if READY:
    evaluate("--bench", "ml/eval/results/latency_kaggle_cpu.json")
    final = json.loads(pathlib.Path("ml/eval/results/inference_params.json").read_text())
    assert final["run_dir"] == params["run_dir"], "the DEV backbone choice changed; re-run step 9"
    run(
        "ml.export.export",
        "--run-dir",
        str(CHOSEN),
        "--params",
        "ml/eval/results/inference_params.json",
        "--model-eval",
        "ml/eval/results/model_eval.json",
        "--out",
        str(ARTIFACT),
        "--refresh-manifest",
    )

## 12. Results bundle for the repository (int8 artifact only; no training weights, no secrets)

The bundle and a copy of `ml/eval/results` are written to new, time-stamped names in
`compliance-runs`. Finish with the output report, then download the bundle (Output panel) or save
a version (KAGGLE.md).

In [ ]:
if READY:
    BUNDLE_OUT = RUNS / f"results-bundle-{EXPECTED_COMMIT[:12]}-{rt.stamp()}.tar"
    rt.require_output_room(RUNS, rt.FINAL_BYTES["bundle"], "results bundle", OUTPUT_LIMIT)
    args = [
        "ml.colab.bundle",
        "pack",
        "--out",
        str(BUNDLE_OUT),
        "--data",
        str(DATA),
        "--baselines",
        str(BASELINES),
        "--results",
        "ml/eval/results",
        "--artifact",
        str(ARTIFACT),
    ]
    for name in ("deberta_v3_base", "modernbert_base", "deberta_v3_base_unseen4"):
        args += ["--run", f"{name}={RUNS / name}"]
    run(*args)
    rt.copy_verified(
        REPO / "ml/eval/results", RUNS / f"eval-results-{EXPECTED_COMMIT[:12]}-{rt.stamp()}"
    )
    crit = json.loads(pathlib.Path("ml/eval/results/baselines_comparison.json").read_text())[
        "handoff_criterion"
    ]
    print("handoff criterion (model vs TF-IDF macro-F1 on TEST):", crit)
    print("download", BUNDLE_OUT, "then run locally: make import-colab-bundle BUNDLE=<path>")
rt.output_report(RUNS, OUTPUT_LIMIT, SCRATCH)